# Notebook 7: Drake Passage — Combined Timeseries

**Kinetic Energy Trends and Eddy Saturation in the Southern Ocean**  
Cristina Marti-Solana, Simon Ruiz, Barbara Barcelo-Llull, Vincent Combes and Ananda Pascual  
Contact: cmarti@imedea.uib-csic.es

---

## Purpose

Co-plot monthly timeseries and linear trends of six key quantities at / near Drake Passage
to assess linked variability among dynamics, fronts, forcing, and transport:

| Panel | Variable | Source |
|---|---|---|
| (a) | **Volume transport** across Drake Passage | Altimetry (Notebook 06) |
| (b) | **EKE** - Drake box | drake_passage_energy_timeseries.csv |
| (c) | **KE** - Drake box | drake_passage_energy_timeseries.csv |
| (d) | **Frontal position** (median latitude) - Atlantic sector | sector_timeseries.csv |
| (e) | **Envelope width** (mean width) - Atlantic sector | sector_timeseries.csv |
| (f) | **Zonal wind stress** - Atlantic sector | wind_stress_sector_timeseries.csv |

> **Note:** Transport is the **surface proxy** (m² s⁻¹) derived from altimetry geostrophic
> velocities (Notebook 06). It is compared to ADCP observations using normalised anomalies.
> Transport data must be generated by Notebook 06 first and saved to
> `outputs/trends/drake_passage_transport.csv`.


## 1. Imports & Paths

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Rectangle
from pathlib import Path
from scipy import stats
import cartopy.crs as ccrs
import cartopy.feature as cfeature

warnings.filterwarnings('ignore', category=RuntimeWarning)

REPO_ROOT  = Path(os.path.abspath(os.path.join(os.getcwd(), '..')))
TRENDS_DIR = REPO_ROOT / 'outputs' / 'trends'
PLOT_DIR   = REPO_ROOT / 'outputs' / 'plots'
FIG_DIR    = REPO_ROOT / 'figures' / 'manuscript'   # publication figures (tracked by git)
FIG_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Style (match notebook 03)
plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Helvetica Neue', 'Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size':         10,
    'axes.labelsize':    11,
    'axes.titlesize':    11,
    'axes.titleweight':  'bold',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   8,
    'legend.framealpha': 0.9,
    'legend.edgecolor':  '#cccccc',
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'axes.grid':         True,
    'grid.alpha':        0.18,
    'grid.linewidth':    0.5,
    'axes.facecolor':    'white',
    'figure.facecolor':  'white',
    'axes.linewidth':    0.6,
})

# Colour palette
C_TRANSPORT     = '#1f77b4'   # blue (altimetry transport)
C_TRANSPORT_OBS = '#264653'   # deep slate (observed transport)
C_EKE           = '#2a9d8f'   # teal
C_KE            = '#e96c6a'   # gold
C_FRONT         = '#6d597a'   # slate-purple
C_WIND          = "#d36b20"   # burnt-orange

print(f'TRENDS_DIR : {TRENDS_DIR}')
print(f'PLOT_DIR   : {PLOT_DIR}')

TRENDS_DIR : /sessions/laughing-compassionate-darwin/mnt/OSR11/repository/outputs/trends
PLOT_DIR   : /sessions/laughing-compassionate-darwin/mnt/OSR11/repository/outputs/plots


## 2. Helper Functions

In [2]:
def running_mean(series, window=12):
    """Centred rolling mean; returns same length as input."""
    return pd.Series(series).rolling(window, center=True, min_periods=window // 2).mean().values


def robust_ylim(y, quantile=95, pad=0.2):
    """Symmetric y-limits from robust spread, useful for anomaly panels."""
    y = np.asarray(y, dtype=float)
    m = np.nanpercentile(np.abs(y[np.isfinite(y)]), quantile) if np.isfinite(y).any() else 1.0
    m = max(m, 1e-6)
    return -(1.0 + pad) * m, (1.0 + pad) * m


def theil_sen_summary(x, y):
    """Theil-Sen trend. Returns (slope, intercept, low_slope, high_slope)."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 6:
        return np.nan, np.nan, np.nan, np.nan
    res = stats.theilslopes(y[mask], x[mask], alpha=0.95)
    return float(res.slope), float(res.intercept), float(res.low_slope), float(res.high_slope)


def mann_kendall(y):
    """Mann-Kendall test with lag-1 autocorrelation variance correction; returns (tau, p_value)."""
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    n = len(y)
    if n < 8:
        return np.nan, np.nan

    s = 0.0
    for i in range(n - 1):
        s += np.sign(y[i + 1:] - y[i]).sum()

    _, counts = np.unique(y, return_counts=True)
    tie_term = np.sum(counts * (counts - 1) * (2 * counts + 5))
    var_s = (n * (n - 1) * (2 * n + 5) - tie_term) / 18.0
    if var_s <= 0:
        return np.nan, np.nan

    # Lag-1 autocorrelation correction via effective sample size.
    y_anom = y - np.nanmean(y)
    den = np.sum(y_anom[:-1] ** 2)
    if den > 0:
        r1 = float(np.sum(y_anom[:-1] * y_anom[1:]) / den)
        r1 = float(np.clip(r1, -0.99, 0.99))
    else:
        r1 = 0.0

    n_eff = n * (1.0 - r1) / (1.0 + r1)
    n_eff = float(np.clip(n_eff, 3.0, float(n)))
    var_s = var_s * (n / n_eff)

    if s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0.0

    p = 2.0 * stats.norm.sf(np.abs(z))
    tau = s / (0.5 * n * (n - 1))
    return float(tau), float(p)


def _mk_sig(pv):
    if not np.isfinite(pv):
        return ''
    if pv < 0.01:
        return '**'
    if pv < 0.05:
        return '*'
    return ''


def annotate_theil_sen(ax, x, y, color, unit, decade_scale=10.0, label_prefix='TS'):
    """Draw Theil-Sen trend and annotate Mann-Kendall p-value in legend."""
    sl, ic, _, _ = theil_sen_summary(x, y)
    if not np.isfinite(sl):
        return

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x_fit = x[mask]
    _, pv = mann_kendall(y[mask])
    sig = _mk_sig(pv)
    ax.plot(
        x_fit,
        sl * x_fit + ic,
        '--',
        color=color,
        lw=1.6,
        zorder=4,
        label=f'{label_prefix}: {sl * decade_scale:+.3f} {unit}/10yr  (MK p={pv:.4f}){sig}'
    )


def drake_window_mask(x):
    """Mask values to Drake Passage time coverage when transport is available."""
    x = np.asarray(x, dtype=float)
    transport_ok = bool(globals().get('_TRANSPORT_AVAILABLE', False))
    t_transport = globals().get('t_tr', np.array([]))
    if transport_ok and len(t_transport) > 0:
        return np.isfinite(x) & (x >= np.nanmin(t_transport)) & (x <= np.nanmax(t_transport))
    return np.isfinite(x)


print('Helper functions defined (Theil-Sen + Mann-Kendall).')

Helper functions defined (Theil-Sen + Mann-Kendall).


## 3. Load KE & EKE — Atlantic Sector

In [3]:
energy_path = TRENDS_DIR / 'drake_passage_energy_timeseries.csv'
if not energy_path.exists():
    raise FileNotFoundError(f'Missing Drake energy file: {energy_path}')

df_energy_drake = pd.read_csv(energy_path)
if 'decimal_year' not in df_energy_drake.columns:
    if 'year' in df_energy_drake.columns and 'month' in df_energy_drake.columns:
        df_energy_drake['decimal_year'] = df_energy_drake['year'] + (df_energy_drake['month'] - 0.5) / 12.0
    else:
        raise ValueError('drake_passage_energy_timeseries.csv requires decimal_year or (year, month).')

df_energy_drake = df_energy_drake.sort_values('decimal_year').reset_index(drop=True)
t_ke = df_energy_drake['decimal_year'].values.astype(float)
ke_gl_ref = df_energy_drake['mean_ke'].values.astype(float) * 1e4
eke_gl_ref = df_energy_drake['mean_eke'].values.astype(float) * 1e4

print(f'Drake energy timeseries: {len(df_energy_drake)} months  ({df_energy_drake["year"].min()}–{df_energy_drake["year"].max()})')
print(f'  Mean EKE = {np.nanmean(eke_gl_ref):.3f} cm² s⁻²')
print(f'  Mean KE  = {np.nanmean(ke_gl_ref):.3f} cm² s⁻²')

Drake energy timeseries: 300 months  (2001–2025)
  Mean EKE = 126.755 cm² s⁻²
  Mean KE  = 351.513 cm² s⁻²


## 4. Load Wind Stress — Atlantic Sector

In [4]:
wind_path = TRENDS_DIR / 'wind_stress_sector_timeseries.csv'
wind_all  = pd.read_csv(wind_path)

wind_atl = wind_all[wind_all['sector'] == 'Atlantic'].copy()
wind_atl['decimal_year'] = wind_atl['year'] + (wind_atl['month'] - 0.5) / 12.0
wind_atl = wind_atl.sort_values('decimal_year').reset_index(drop=True)

t_wind   = wind_atl['decimal_year'].values
tau_x    = wind_atl['tau_x'].values

print(f'Wind stress timeseries: {len(wind_atl)} months  '
      f'({wind_atl["year"].min()}–{wind_atl["year"].max()})')
print(f'  Mean τx = {tau_x.mean():.3f}')

Wind stress timeseries: 276 months  (2000–2022)
  Mean τx = 5.470


## 5. Load Frontal Position, Envelope Width, and Transport

Frontal metrics are read from sector_timeseries.csv (Atlantic sector).
Transport is computed in **Notebook 06** using CMEMS altimetry and saved to
outputs/trends/drake_passage_transport.csv.

Expected frontal columns: median_lat (or mean_lat), mean_width_km (or mean_envelope_span).
Expected transport columns: year, month, decimal_year (or derivable), transport (Sv).

In [5]:
# ── Drake Passage altimetry surface transport (from Notebook 06) ────────────
transport_path = TRENDS_DIR / 'drake_passage_transport.csv'
_TRANSPORT_AVAILABLE = transport_path.exists()

if _TRANSPORT_AVAILABLE:
    df_tr = pd.read_csv(transport_path)

    if 'decimal_year' not in df_tr.columns:
        df_tr['decimal_year'] = df_tr['year'] + (df_tr['month'] - 0.5) / 12.0

    df_tr = df_tr.sort_values('decimal_year').reset_index(drop=True)
    t_tr  = df_tr['decimal_year'].values

    # Altimetry transport is stored as transport_m2s (m² s⁻¹)
    if 'transport_m2s' in df_tr.columns:
        sv = df_tr['transport_m2s'].values
        _transport_units = 'm² s⁻¹'
        transport_col = 'transport_m2s'
    else:
        raise ValueError(
            'Notebook 7 requires altimetry transport in transport_m2s column. '
            'Re-run Notebook 06 to regenerate drake_passage_transport.csv.'
        )

    print(f'Transport timeseries (altimetry surface proxy, {transport_col}): '
          f'{len(df_tr)} months  ({df_tr["year"].min()}–{df_tr["year"].max()})')
    print(f'  Mean T_surf = {np.nanmean(sv):.2f} {_transport_units}')
else:
    print('Transport CSV not found.')
    print(f'  Expected: {transport_path}')
    print('  → Run Notebook 06 first.')
    t_tr = np.array([])
    sv   = np.array([])
    _transport_units = 'm² s⁻¹'
    transport_col = 'transport_m2s'

# ── Observed Drake Passage total transport (ADCP) ────────────────────────
obs_transport_path = TRENDS_DIR / 'drake_passage_total_transport_observations.csv'
_OBS_TRANSPORT_AVAILABLE = False
obs_col    = None
obs_source = None

if obs_transport_path.exists():
    df_tr_obs = pd.read_csv(obs_transport_path)
    if 'decimal_year' not in df_tr_obs.columns:
        if 'date' in df_tr_obs.columns:
            dt = pd.to_datetime(df_tr_obs['date'])
            df_tr_obs['decimal_year'] = dt.dt.year + (dt.dt.dayofyear - 0.5) / 365.25
            df_tr_obs['year'] = dt.dt.year
        else:
            raise ValueError('Observed transport file requires date or decimal_year column.')

    obs_candidates = ['total_transport_sv', 'transport_sv']
    obs_col = next((c for c in obs_candidates if c in df_tr_obs.columns), None)

    if obs_col is None:
        if 'total_transport_m3s' in df_tr_obs.columns:
            df_tr_obs['total_transport_sv'] = df_tr_obs['total_transport_m3s'] / 1e6
            obs_col = 'total_transport_sv'

    if obs_col is not None:
        _OBS_TRANSPORT_AVAILABLE = True
        obs_source = f'ADCP file ({obs_col})'
        df_tr_obs = df_tr_obs.sort_values('decimal_year').reset_index(drop=True)
        t_tr_obs  = df_tr_obs['decimal_year'].values.astype(float)
        sv_obs    = df_tr_obs[obs_col].values.astype(float)
        yr_min_obs = int(np.floor(np.nanmin(t_tr_obs))) if len(t_tr_obs) else np.nan
        yr_max_obs = int(np.floor(np.nanmax(t_tr_obs))) if len(t_tr_obs) else np.nan
        print(f'Transport observations ({obs_col}): {len(df_tr_obs)} samples  '
              f'({yr_min_obs}–{yr_max_obs})')
        print(f'  Mean observed transport = {np.nanmean(sv_obs):.1f} Sv')
    else:
        print(f'No suitable transport column in {obs_transport_path}')
        t_tr_obs = np.array([])
        sv_obs   = np.array([])
else:
    print(f'Observed transport CSV not found: {obs_transport_path}')
    t_tr_obs = np.array([])
    sv_obs   = np.array([])


Transport timeseries (altimetry surface proxy, transport_m2s): 288 months  (2001–2024)
  Mean T_surf = 78205.56 m² s⁻¹
Transport observations (total_transport_sv): 248 samples  (2005–2019)
  Mean observed transport = 75.7 Sv


In [6]:
# Drake Passage KE/EKE (same box as Notebook 06)
energy_path = TRENDS_DIR / 'drake_passage_energy_timeseries.csv'
if not energy_path.exists():
    raise FileNotFoundError(f'Missing Drake energy file: {energy_path}')

df_energy_drake = pd.read_csv(energy_path)
if 'decimal_year' not in df_energy_drake.columns:
    if 'year' in df_energy_drake.columns and 'month' in df_energy_drake.columns:
        df_energy_drake['decimal_year'] = df_energy_drake['year'] + (df_energy_drake['month'] - 0.5) / 12.0
    else:
        raise ValueError('drake_passage_energy_timeseries.csv requires decimal_year or (year, month).')

df_energy_drake = df_energy_drake.sort_values('decimal_year').reset_index(drop=True)
t_ke = df_energy_drake['decimal_year'].values.astype(float)
ke_gl_ref = (df_energy_drake['mean_ke'].values.astype(float) * 1e4)
eke_gl_ref = (df_energy_drake['mean_eke'].values.astype(float) * 1e4)

print(f'Drake energy timeseries: {len(df_energy_drake)} months  ({df_energy_drake["year"].min()}-{df_energy_drake["year"].max()})')
print(f'  Mean KE  = {np.nanmean(ke_gl_ref):.3f} cm² s⁻²')
print(f'  Mean EKE = {np.nanmean(eke_gl_ref):.3f} cm² s⁻²')

Drake energy timeseries: 300 months  (2001-2025)
  Mean KE  = 351.513 cm² s⁻²
  Mean EKE = 126.755 cm² s⁻²


## 6. Publication Figure: Drake Map + Transport(+Wind) + KE/EKE Trends

Single composite figure for publication with four panels in a compact 2x2 layout.

Region strategy used in this figure:

1. Map: shows the Drake transect used for transport and one common analysis box
2. Wind, KE and EKE trends: all use the same common analysis box
3. Region is marked only on the map (not repeated in subplots)

Trend slopes and Mann-Kendall significance are annotated inside the trend subplots (Notebook 3a style).

In [7]:
import xarray as xr

# Publication figure variant: True -> 'drake_passage_ke_eke.png' (KE + EKE in panel d),
# False -> 'drake_passage_eke.png' (EKE only). Both variants are used; re-run this cell with
# the other value to produce the second file.
INCLUDE_KE = True

# Common analysis box for wind + KE + EKE (Drake-focused)
ANALYSIS_LON_MIN, ANALYSIS_LON_MAX = -80.0, -52.0
ANALYSIS_LAT_MIN, ANALYSIS_LAT_MAX = -63.0, -54.0

# Exact transport section used in Notebook 06 (oblique transect)
TRANSECT_LON_START, TRANSECT_LAT_START = -65.0, -55.0
TRANSECT_LON_END, TRANSECT_LAT_END = -60.0, -63.0
DRAKE_LON_MIN = min(TRANSECT_LON_START, TRANSECT_LON_END)
DRAKE_LON_MAX = max(TRANSECT_LON_START, TRANSECT_LON_END)
DRAKE_LAT_SOUTH = min(TRANSECT_LAT_START, TRANSECT_LAT_END)
DRAKE_LAT_NORTH = max(TRANSECT_LAT_START, TRANSECT_LAT_END)

# Helpers local to this publication figure

def _trend_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 8:
        return {'slope_yr': np.nan, 'intercept': np.nan, 'mk_p': np.nan}
    ts = stats.theilslopes(y[m], x[m], alpha=0.95)
    _, p = mann_kendall(y[m])
    return {'slope_yr': float(ts.slope), 'intercept': float(ts.intercept), 'mk_p': float(p)}


def _box_series_from_monthlies(monthly_dir, var_name, out_name, scale=1.0):
    rows = []
    files = sorted(monthly_dir.glob('*/*.nc'))
    if not files:
        return pd.DataFrame(columns=['year', 'month', 'decimal_year', out_name, 'n_valid'])

    for fp in files:
        try:
            yr = int(fp.parent.name)
            mo = int(fp.stem)
        except Exception:
            continue

        try:
            with xr.open_dataset(fp) as ds:
                if var_name not in ds:
                    continue

                lon_name = 'longitude' if 'longitude' in ds.coords else ('lon' if 'lon' in ds.coords else None)
                lat_name = 'latitude' if 'latitude' in ds.coords else ('lat' if 'lat' in ds.coords else None)
                if lon_name is None or lat_name is None:
                    continue

                lon = ds[lon_name].values.astype(float)
                lat = ds[lat_name].values.astype(float)
                fld = ds[var_name].values.astype(float)

                if np.nanmax(lon) > 180:
                    lon_min = ANALYSIS_LON_MIN % 360.0
                    lon_max = ANALYSIS_LON_MAX % 360.0
                else:
                    lon_min = ANALYSIS_LON_MIN
                    lon_max = ANALYSIS_LON_MAX

                m_lon = (lon >= lon_min) & (lon <= lon_max)
                m_lat = (lat >= ANALYSIS_LAT_MIN) & (lat <= ANALYSIS_LAT_MAX)

                if (not np.any(m_lon)) or (not np.any(m_lat)):
                    val_mean = np.nan
                    n_valid = 0
                else:
                    sub = fld[np.ix_(m_lat, m_lon)]
                    n_valid = int(np.isfinite(sub).sum())
                    val_mean = float(np.nanmean(sub) * scale) if n_valid > 0 else np.nan

                rows.append({
                    'year': yr,
                    'month': mo,
                    'decimal_year': yr + (mo - 0.5) / 12.0,
                    out_name: val_mean,
                    'n_valid': n_valid,
                })
        except Exception:
            continue

    out = pd.DataFrame(rows)
    if len(out):
        out = out.sort_values('decimal_year').reset_index(drop=True)
    return out


def _trend_text(label, st, unit_per_year):
    if not np.isfinite(st['slope_yr']):
        return f'{label}: N/A'
    sig = _mk_sig(st['mk_p'])
    return f"{label}: {st['slope_yr']:+.3f} {unit_per_year} (p={st['mk_p']:.4f}){sig}"


def _slope_ci(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 8:
        return np.nan, np.nan, np.nan
    ts = stats.theilslopes(y[m], x[m], alpha=0.95)
    return float(ts.slope), float(ts.low_slope), float(ts.high_slope)


def _center_series(y):
    y = np.asarray(y, dtype=float)
    m = np.isfinite(y)
    if m.sum() < 2:
        return y
    return y - np.nanmean(y[m])


def _annotate_bar_sig(ax, xpos, slopes, err_low, err_high, pvals):
    all_slopes = np.asarray(slopes, dtype=float)
    yr = np.nanmax(np.abs(all_slopes)) if np.isfinite(all_slopes).any() else 1.0
    yr = max(float(yr), 1e-6)
    for xi, s, el, eh, p in zip(xpos, slopes, err_low, err_high, pvals):
        st = _mk_sig(p)
        if (not st) or (not np.isfinite(s)):
            continue
        el = float(el) if np.isfinite(el) else 0.0
        eh = float(eh) if np.isfinite(eh) else 0.0
        e = eh if s >= 0 else el
        ytxt = s + e + 0.01 * yr if s >= 0 else s - e - 0.01 * yr
        ax.text(
            xi,
            ytxt,
            st,
            ha='center',
            va='bottom' if s >= 0 else 'top',
            fontsize=10.5,
            fontweight='semibold',
            color='#1F2933',
        )


# -----------------------------------------------------------------
# Data for publication figure
# -----------------------------------------------------------------

# Transport (altimetry surface proxy + observed total transport)
t_gl = np.asarray(t_tr, dtype=float) if len(t_tr) else np.array([])
sv_gl = np.asarray(sv, dtype=float) if len(sv) else np.array([])

# Use altimetry surface transport from Notebook 06 output.
# Altimetry surface transport proxy
if ('df_tr' in globals()) and isinstance(df_tr, pd.DataFrame) and 'transport_m2s' in df_tr.columns:
    sv_gl = df_tr['transport_m2s'].values.astype(float)
    selected_transport_col = 'transport_m2s'
elif len(t_tr) > 0:
    sv_gl = np.asarray(sv, dtype=float)
    selected_transport_col = 'transport_m2s'
else:
    raise ValueError('Notebook 7 requires altimetry transport in transport_m2s. Re-run Notebook 06.')
print(f'Using altimetry surface transport column: {selected_transport_col}')

# Drake-box altimetry KE/EKE from energy timeseries CSV
if 'df_energy_drake' not in globals():
    energy_path = TRENDS_DIR / 'drake_passage_energy_timeseries.csv'
    if not energy_path.exists():
        raise FileNotFoundError(f'Missing Drake energy file: {energy_path}')
    df_energy_drake = pd.read_csv(energy_path)
    if 'decimal_year' not in df_energy_drake.columns:
        if 'year' in df_energy_drake.columns and 'month' in df_energy_drake.columns:
            df_energy_drake['decimal_year'] = df_energy_drake['year'] + (df_energy_drake['month'] - 0.5) / 12.0
        else:
            raise ValueError('drake_passage_energy_timeseries.csv requires decimal_year or (year, month).')
    df_energy_drake = df_energy_drake.sort_values('decimal_year').reset_index(drop=True)

t_ke = df_energy_drake['decimal_year'].values.astype(float)
ke_gl_ref = df_energy_drake['mean_ke'].values.astype(float) * 1e4
eke_gl_ref = df_energy_drake['mean_eke'].values.astype(float) * 1e4

# Observed Drake Passage transport (0-760 m target)
t_obs = np.asarray(t_tr_obs, dtype=float) if 't_tr_obs' in globals() else np.array([])
sv_obs2 = np.asarray(sv_obs, dtype=float) if 'sv_obs' in globals() else np.array([])
if len(t_obs) == 0:
    print('No observed 0-760 m transport series loaded; observed curve will be skipped.')

# Wind (same common analysis box)
t_wind_pub = np.asarray(t_wind, dtype=float) if 't_wind' in globals() else np.array([])
tau_x_pub = np.asarray(tau_x, dtype=float) if 'tau_x' in globals() else np.array([])
if len(t_wind_pub):
    m_wind = (
        np.isfinite(t_wind_pub)
        & np.isfinite(tau_x_pub)
        & (t_wind_pub >= 2001.0)
        & (t_wind_pub < 2026.0)
    )
    t_wind_pub = t_wind_pub[m_wind]
    tau_x_pub = tau_x_pub[m_wind]

# Altimetry KE/EKE from monthly maps, now using the SAME common analysis box
hp_dir = REPO_ROOT / 'outputs' / 'monthly_half-power_points'
df_eke_alt = _box_series_from_monthlies(hp_dir, 'eke', 'eke_cm2', scale=1e4)
df_ke_alt = _box_series_from_monthlies(hp_dir, 'ke', 'ke_cm2', scale=1e4)

if df_eke_alt.empty:
    df_eke_alt = pd.DataFrame({'decimal_year': t_ke, 'eke_cm2': eke_gl_ref})
    print('Warning: Common-box altimetry EKE not found; fallback to Drake energy reference.')

if df_ke_alt.empty:
    df_ke_alt = pd.DataFrame({'decimal_year': t_ke, 'ke_cm2': ke_gl_ref})
    print('Warning: Common-box altimetry KE not found; fallback to Drake energy reference.')

t_alt = df_eke_alt['decimal_year'].values.astype(float)
eke_alt_raw = df_eke_alt['eke_cm2'].values.astype(float)
t_alt_ke = df_ke_alt['decimal_year'].values.astype(float)
ke_alt_raw = df_ke_alt['ke_cm2'].values.astype(float)

# Adjust altimetry level to reference mean for region-consistent visual comparison
m_ov_eke = np.isfinite(t_ke) & np.isfinite(eke_gl_ref)
if np.isfinite(eke_alt_raw).any() and m_ov_eke.any():
    eke_alt = eke_alt_raw - np.nanmean(eke_alt_raw) + np.nanmean(eke_gl_ref)
else:
    eke_alt = eke_alt_raw

m_ov_ke = np.isfinite(t_ke) & np.isfinite(ke_gl_ref)
if np.isfinite(ke_alt_raw).any() and m_ov_ke.any():
    ke_alt = ke_alt_raw - np.nanmean(ke_alt_raw) + np.nanmean(ke_gl_ref)
else:
    ke_alt = ke_alt_raw

# Precompute trend stats for annotations and bars
st_tr_gl = _trend_stats(t_gl, sv_gl) if len(t_gl) else {'slope_yr': np.nan, 'intercept': np.nan, 'mk_p': np.nan}
st_tr_obs = _trend_stats(t_obs, sv_obs2) if len(t_obs) else {'slope_yr': np.nan, 'intercept': np.nan, 'mk_p': np.nan}
st_wind_pub = _trend_stats(t_wind_pub, tau_x_pub) if len(t_wind_pub) else {'slope_yr': np.nan, 'intercept': np.nan, 'mk_p': np.nan}
st_eke_gl = _trend_stats(t_ke, eke_gl_ref)
st_eke_alt = _trend_stats(t_alt, eke_alt)
st_ke_gl = _trend_stats(t_ke, ke_gl_ref)
st_ke_alt = _trend_stats(t_alt_ke, ke_alt)

# Transport anomalies (remove mean level to compare variability on one axis).
sv_gl_anom = _center_series(sv_gl) if len(sv_gl) else sv_gl
sv_obs_anom = _center_series(sv_obs2) if len(sv_obs2) else sv_obs2

def _zscore(series):
    series = np.asarray(series, dtype=float)
    valid = np.isfinite(series)
    if valid.sum() < 2:
        return series
    sigma = np.nanstd(series[valid])
    if not np.isfinite(sigma) or sigma == 0:
        return np.zeros_like(series, dtype=float)
    return (series - np.nanmean(series[valid])) / sigma

sv_gl_z = _zscore(sv_gl)
sv_obs_z = _zscore(sv_obs2)
st_tr_gl_std = _trend_stats(t_gl, sv_gl_z) if len(t_gl) else {'slope_yr': np.nan, 'intercept': np.nan, 'mk_p': np.nan}
st_tr_obs_std = _trend_stats(t_obs, sv_obs_z) if len(t_obs) else {'slope_yr': np.nan, 'intercept': np.nan, 'mk_p': np.nan}

# -----------------------------------------------------------------
# Figure layout (single publication figure)
# -----------------------------------------------------------------
fig = plt.figure(figsize=(10, 10))
outer = fig.add_gridspec(
    3, 2,
    height_ratios=[1.2, 1, 1.2],
    width_ratios=[0.5, 1],
    hspace=0.2,
    wspace=0.2,
)

# ============================================================
# (a) MAP
# ============================================================
axm = fig.add_subplot(outer[0, 0], projection=ccrs.SouthPolarStereo())
axm.set_extent([-88, -48, -69, -50], crs=ccrs.PlateCarree())
axm.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#111111', edgecolor='#111111')
axm.add_feature(cfeature.COASTLINE.with_scale('50m'), linewidth=0.45, color='#111111')
for spine in axm.spines.values():
    spine.set_visible(False)
if hasattr(axm, 'outline_patch'):
    axm.outline_patch.set_visible(False)
axm.gridlines(draw_labels=False, linewidth=0.35, color='#7f8a96', alpha=0.45, linestyle='--')
axm.add_patch(Rectangle(
    (ANALYSIS_LON_MIN, ANALYSIS_LAT_MIN),
    ANALYSIS_LON_MAX - ANALYSIS_LON_MIN,
    ANALYSIS_LAT_MAX - ANALYSIS_LAT_MIN,
    fill=False, lw=1.6, ec='#2a9d8f', transform=ccrs.PlateCarree(), label='Analysis box',
))
axm.plot([TRANSECT_LON_START, TRANSECT_LON_END],
         [TRANSECT_LAT_START, TRANSECT_LAT_END],
         color=C_TRANSPORT, lw=3.0, linestyle='--',
         transform=ccrs.PlateCarree(), label='Transport section')
axm.legend(loc='lower left', fontsize=8, framealpha=0)

# ============================================================
# (b) TREND BARS (SEPARATED BY VARIABLE)
# ============================================================
bar_grid = outer[0, 1].subgridspec(1, 2, wspace=0.5, width_ratios=[1, 1])
ax_bar_tw = fig.add_subplot(bar_grid[0, 0])
ax_bar_en = fig.add_subplot(bar_grid[0, 1])

# Transport bars (standardized slope / yr)
tw_labels = ['Altimetry', 'In-situ']
tw_base_colors = [C_TRANSPORT, C_TRANSPORT_OBS]
tw_series = [(t_gl, sv_gl_z), (t_obs, sv_obs_z)]
tw_sl, tw_lo, tw_hi = [], [], []
for x, y in tw_series:
    s, l, h = _slope_ci(x, y)
    tw_sl.append(s)
    tw_lo.append(max(s - l, 0.0) if np.isfinite(s) and np.isfinite(l) else np.nan)
    tw_hi.append(max(h - s, 0.0) if np.isfinite(s) and np.isfinite(h) else np.nan)
tw_pv = [st_tr_gl_std['mk_p'], st_tr_obs_std['mk_p']]
tw_colors = [bc if (np.isfinite(pv) and pv < 0.05) else '#C9CDD1' for bc, pv in zip(tw_base_colors, tw_pv)]
xx = np.arange(len(tw_labels), dtype=float)
ax_bar_tw.bar(xx, tw_sl, width=0.62, color=tw_colors, alpha=0.9)
ax_bar_tw.errorbar(xx, tw_sl, yerr=[tw_lo, tw_hi], fmt='none', ecolor='black', capsize=3)
ax_bar_tw.axhline(0.0, color='gray', linestyle='--', lw=1)
ax_bar_tw.set_xticks(xx)
ax_bar_tw.tick_params(axis='x', colors="white")
ax_bar_tw.set_xticklabels(tw_labels, fontsize=8, color="k", fontweight='bold')
ax_bar_tw.set_ylabel('Standardized transport slope (σ/yr)', fontsize=8)
ax_bar_tw.grid(axis='y', alpha=0.2)
ax_bar_tw.spines["bottom"].set_visible(False)
_annotate_bar_sig(ax_bar_tw, xx, tw_sl, tw_lo, tw_hi, tw_pv)

# Energy bars (altimetry only, cm^2 s^-2 / yr)
en_labels = ['EKE', 'KE']
en_base_colors = [C_EKE, C_KE]
en_series = [(t_alt, eke_alt), (t_alt_ke, ke_alt)]
en_sl, en_lo, en_hi = [], [], []
for x, y in en_series:
    s, l, h = _slope_ci(x, y)
    en_sl.append(s)
    en_lo.append(max(s - l, 0.0) if np.isfinite(s) and np.isfinite(l) else np.nan)
    en_hi.append(max(h - s, 0.0) if np.isfinite(s) and np.isfinite(h) else np.nan)
en_pv = [st_eke_alt['mk_p'], st_ke_alt['mk_p']]
en_neutral = ['#D5DADF', '#C9CDD1']
en_colors = [bc if (np.isfinite(pv) and pv < 0.05) else nc for bc, nc, pv in zip(en_base_colors, en_neutral, en_pv)]
yy_en = np.arange(len(en_labels), dtype=float)
ax_bar_en.bar(yy_en, en_sl, width=0.62, color=en_colors, alpha=0.9)
ax_bar_en.errorbar(yy_en, en_sl, yerr=[en_lo, en_hi], fmt='none', ecolor='black', capsize=3)
ax_bar_en.axhline(0.0, color='gray', linestyle='--', lw=1)
ax_bar_en.set_xticks(yy_en)
ax_bar_en.tick_params(axis='x', colors="white")
ax_bar_en.set_xticklabels(en_labels, fontsize=8, color="k", fontweight='bold')
ax_bar_en.set_ylabel(r'cm$^2$ s$^{-2}$/yr', fontsize=8)
ax_bar_en.grid(axis='y', alpha=0.2)
ax_bar_en.spines["bottom"].set_visible(False)
_annotate_bar_sig(ax_bar_en, yy_en, en_sl, en_lo, en_hi, en_pv)

# ============================================================
# (c) TRANSPORT ANOMALIES + WIND (WIDE)
# ============================================================
ax_tr = fig.add_subplot(outer[1, :])

if len(t_gl):
    ax_tr.plot(t_gl, sv_gl_z, color=C_TRANSPORT, alpha=0.15, lw=1.0)
    ax_tr.plot(t_gl, running_mean(sv_gl_z), color=C_TRANSPORT, lw=3.0, label='Altimetry')

if len(t_obs):
    ax_tr.plot(t_obs, running_mean(sv_obs_z), color=C_TRANSPORT_OBS, lw=2.2, label='In-situ')

ax_tr_w = ax_tr.twinx()
if len(t_wind_pub):
    ax_tr_w.plot(t_wind_pub, running_mean(tau_x_pub), color=C_WIND, lw=1.2, alpha=0.55, linestyle='--', label='Wind')

ax_tr.axhline(0.0, color='#6b7280', lw=0.9, ls='--', alpha=0.85)
ax_tr.set_ylabel('Standardized transport anomaly (σ)')
ax_tr.tick_params(axis='y', colors='black')
ax_tr.spines['left'].set_color('black')
ax_tr.spines['left'].set_linewidth(1.0)

ax_tr_w.set_ylabel(r'Zonal Wind Stress (N m$^{-2}$)', color=C_WIND)
ax_tr_w.yaxis.set_label_position('right')
ax_tr_w.yaxis.tick_right()
ax_tr_w.tick_params(axis='y', colors=C_WIND)
ax_tr_w.spines['right'].set_visible(True)
ax_tr_w.spines['right'].set_color(C_WIND)
ax_tr_w.spines['right'].set_linewidth(1.2)

ax_tr.set_xlabel('')
ax_tr.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax_tr.tick_params(axis='x', labelbottom=False)
ax_tr.grid(alpha=0.15)

h1, l1 = ax_tr.get_legend_handles_labels()
h2, l2 = ax_tr_w.get_legend_handles_labels()
ax_tr.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=8, frameon=False)

# Add trend text for transport and wind directly on panel (c)
transport_wind_text = '\n'.join([
    _trend_text('Transport Alt (std)', st_tr_gl_std, 'sigma/yr'),
    _trend_text('Transport In-situ (std)', st_tr_obs_std, 'sigma/yr') if len(t_obs) else 'Transport In-situ: N/A'])
ax_tr.text(0.01, 0.03, transport_wind_text, transform=ax_tr.transAxes, fontsize=7.1, color='#1f2937')

# ============================================================
# (d) ALTIMETRY EKE + KE (broken y-axis, monthly + smoothed)
# ============================================================
# Create a small sub-gridspec to stack two axes and produce a "broken" y-axis effect
ax_ke = fig.add_subplot(outer[2, :])

# Smoothed (running mean) and monthly raw series
ke_smooth = running_mean(ke_alt)
eke_smooth = running_mean(eke_alt)

# Monthly raw thin lines (fainter)
if INCLUDE_KE and len(t_alt_ke):
    ax_ke.plot(t_alt_ke, ke_alt, color=C_KE, lw=0.6, alpha=0.25)
if len(t_alt):
    ax_ke.plot(t_alt, eke_alt, color=C_EKE, lw=0.6, alpha=0.25)

# Smoothed bold lines on respective axes
if INCLUDE_KE:
    ax_ke.plot(t_alt_ke, ke_smooth, color=C_KE, lw=2.2, label='KE')
ax_ke.plot(t_alt, eke_smooth, color=C_EKE, lw=2.2, label='EKE')

# Add explicit trend lines for the KE and EKE time series
if INCLUDE_KE and np.isfinite(st_ke_alt['slope_yr']) and np.isfinite(st_ke_alt['intercept']) and len(t_alt_ke):
    ax_ke.plot(
        t_alt_ke,
        st_ke_alt['slope_yr'] * t_alt_ke + st_ke_alt['intercept'],
        color=C_KE,
        lw=1.8,
        ls='--',
        alpha=0.95,
    )
if np.isfinite(st_eke_alt['slope_yr']) and np.isfinite(st_eke_alt['intercept']) and len(t_alt):
    ax_ke.plot(
        t_alt,
        st_eke_alt['slope_yr'] * t_alt + st_eke_alt['intercept'],
        color=C_EKE,
        lw=1.8,
        ls='--',
        alpha=0.95,
    )

# Compute focused y-limits with padding
def _pad(v, pad=0.06):
    v = np.asarray(v, dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return (-1.0, 1.0)
    r = np.nanmax(v) - np.nanmin(v)
    if r == 0:
        r = abs(np.nanmax(v)) if np.isfinite(np.nanmax(v)) else 1.0
    return (np.nanmin(v) - pad * r, np.nanmax(v) + pad * r)

all_vals = np.concatenate([
    eke_smooth[np.isfinite(eke_smooth)] if np.isfinite(eke_smooth).any() else np.array([]),
    ke_smooth[np.isfinite(ke_smooth)] if np.isfinite(ke_smooth).any() else np.array([]),
])

# Labels, grid and x-axis
ax_ke.set_ylabel(r'Energy (cm$^2$ s$^{-2}$)')
ax_ke.tick_params(axis='y', colors='black')
ax_ke.grid(alpha=0.15)
ax_ke.set_xlabel('Year')
ax_ke.xaxis.set_major_locator(mticker.MultipleLocator(4))

# Combined legend
handles, labels = ax_ke.get_legend_handles_labels()
if handles:
    ax_ke.legend(handles, labels, loc='upper left', fontsize=8, frameon=False)

# Trend text: KE on top, EKE on bottom
ax_ke.text(0.01, 0.03, _trend_text('EKE', st_eke_alt, r'cm$^2$ s$^{-2}$/yr'), transform=ax_ke.transAxes, fontsize=7.1, color='#1f2937')
if INCLUDE_KE:
    ax_ke.text(0.01, 0.09, _trend_text('KE', st_ke_alt, r'cm$^2$ s$^{-2}$/yr'), transform=ax_ke.transAxes, fontsize=7.1, color='#1f2937')

fig.tight_layout()

# Manuscript figure 'drake_passage' (main article)
fig_name = 'drake_passage_ke_eke.png' if INCLUDE_KE else 'drake_passage_eke.png'
out_path = FIG_DIR / fig_name
fig.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

print('\nConcise trend summary for publication figure:')
print(f"  Transport altimetry (std): {st_tr_gl_std['slope_yr']:+.4f} sigma/yr, MK p={st_tr_gl_std['mk_p']:.4f}")
if len(t_obs):
    print(f"  Transport Obs (std): {st_tr_obs_std['slope_yr']:+.4f} sigma/yr, MK p={st_tr_obs_std['mk_p']:.4f}")
else:
    print('  Transport Obs (std): N/A (no observed 0-760 m file loaded)')
print(f"  Wind tau_x       : {st_wind_pub['slope_yr']:+.4f} N m$^{{-2}}$/yr, MK p={st_wind_pub['mk_p']:.4f}")
print(f"  EKE Altimetry   : {st_eke_alt['slope_yr']:+.4f} cm$^2$ s$^{{-2}}$/yr, MK p={st_eke_alt['mk_p']:.4f}")
print(f"  KE Altimetry    : {st_ke_alt['slope_yr']:+.4f} cm$^2$ s$^{{-2}}$/yr, MK p={st_ke_alt['mk_p']:.4f}")

Using altimetry surface transport column: transport_m2s


/sessions/laughing-compassionate-darwin/.local/lib/python3.10/site-packages/cartopy/mpl/feature_artist.py:143: UserWarning: facecolor will have no effect as it has been defined as "never".
  warnings.warn('facecolor will have no effect as it has been '
/sessions/laughing-compassionate-darwin/tmp/ipykernel_11/356185827.py:469: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


Saved: /sessions/laughing-compassionate-darwin/mnt/OSR11/repository/figures/manuscript/drake_passage_ke_eke.png

Concise trend summary for publication figure:
  Transport altimetry (std): +0.0134 sigma/yr, MK p=0.3259
  Transport Obs (std): -0.0127 sigma/yr, MK p=0.4222
  Wind tau_x       : +0.0018 N m$^{-2}$/yr, MK p=0.8700
  EKE Altimetry   : +2.2021 cm$^2$ s$^{-2}$/yr, MK p=0.0159
  KE Altimetry    : +0.7996 cm$^2$ s$^{-2}$/yr, MK p=0.0567


## 7. Save Transport CSV (run from Notebook 06)

If you have already computed the altimetry surface transport in Notebook 06, paste the
following snippet into that notebook (after the integration cell) to save the result:

```python
df_transport['decimal_year'] = df_transport['year'] + (df_transport['month'] - 0.5) / 12.0
save_path = Path(REPO_ROOT) / 'outputs' / 'trends' / 'drake_passage_transport.csv'
df_transport[['year', 'month', 'decimal_year', 'transport_m2s']].to_csv(save_path, index=False)
print(f'Saved: {save_path}')
```


## 8. Trend Summary Table (Drake Section, Theil-Sen + Mann-Kendall)

In [8]:
rows = []

def add_ts_mk_row(name, unit, x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = drake_window_mask(x) & np.isfinite(y)
    if m.sum() < 8:
        rows.append({
            'Variable': name,
            'Units': unit,
            'Theil-Sen / 10yr': 'N/A',
            'MK p-value': 'N/A',
            'Significant': ''
        })
        return

    sl, _, _, _ = theil_sen_summary(x[m], y[m])
    _, pv = mann_kendall(y[m])
    rows.append({
        'Variable': name,
        'Units': unit,
        'Theil-Sen / 10yr': f'{sl*10:+.4f}',
        'MK p-value': f'{pv:.4f}',
        'Significant': 'yes' if pv < 0.05 else ''
    })

add_ts_mk_row('Transport (Drake, altimetry)', 'm² s⁻¹', t_tr, sv if _TRANSPORT_AVAILABLE else np.array([]))
add_ts_mk_row('Transport (Drake, observed total)', 'Sv', t_tr_obs, sv_obs if _OBS_TRANSPORT_AVAILABLE else np.array([]))
add_ts_mk_row('tau_x (Atlantic)', '1e-2 N m-2', t_wind, tau_x)
add_ts_mk_row('KE (Altimetry)', 'cm2 s-2', t_ke, ke_gl_ref)
add_ts_mk_row('EKE (Altimetry)', 'cm2 s-2', t_ke, eke_gl_ref)

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))

                         Variable      Units Theil-Sen / 10yr MK p-value Significant
     Transport (Drake, altimetry)     m² s⁻¹       +1554.2515     0.3259            
Transport (Drake, observed total)         Sv          -1.0575     0.4222            
                 tau_x (Atlantic) 1e-2 N m-2          +0.0183     0.8700            
                   KE (Altimetry)    cm2 s-2         +14.3783     0.0058         yes
                  EKE (Altimetry)    cm2 s-2          +5.9191     0.1164            


## 9. Additional Drake Trend Diagnostics (No Additional Figure)

To keep the notebook output to **one single figure** (Section 6),
this section computes and prints requested Drake diagnostics without creating a new plot:

1. Altimetry Drake Passage surface transport trend
2. Wind stress trend (altimetry-based sector series)
3. Drake Passage EKE and KE trends (from altimetry monthly maps)
4. Transport trend comparison: altimetry vs ADCP observations


In [9]:
import xarray as xr

# Paths and constants
transport_path = TRENDS_DIR / 'drake_passage_transport.csv'
wind_path = TRENDS_DIR / 'wind_stress_sector_timeseries.csv'
hp_dir = REPO_ROOT / 'outputs' / 'monthly_half-power_points'

ADCP_SLOPE_SV_YR = 0.02  # Sv/yr (given)
ADCP_SLOPE_ERR_SV_YR = 0.00
ADCP_P_TEXT = 'p < 0.05'

DRAKE_LON_MIN, DRAKE_LON_MAX = -65.0, -55.0
DRAKE_LAT_MIN, DRAKE_LAT_MAX = -63.0, -52.0

# -----------------------------
# 1) Altimetry surface transport series
# -----------------------------
if not transport_path.exists():
    raise FileNotFoundError(f'Missing transport file: {transport_path}')

df_tr = pd.read_csv(transport_path).copy()
if 'decimal_year' not in df_tr.columns:
    if 'year' in df_tr.columns and 'month' in df_tr.columns:
        df_tr['decimal_year'] = df_tr['year'] + (df_tr['month'] - 0.5) / 12.0
    elif 'time' in df_tr.columns:
        ttmp = pd.to_datetime(df_tr['time'])
        df_tr['year'] = ttmp.dt.year
        df_tr['month'] = ttmp.dt.month
        df_tr['decimal_year'] = df_tr['year'] + (df_tr['month'] - 0.5) / 12.0
    else:
        raise ValueError('Transport file requires either (year, month) or time columns.')

df_tr = df_tr.sort_values('decimal_year').reset_index(drop=True)
t_tr2 = df_tr['decimal_year'].values.astype(float)
if 'transport_m2s' not in df_tr.columns:
    raise ValueError('Section 9 requires altimetry transport_m2s column. Re-run Notebook 06.')
selected_transport_col = 'transport_m2s'
sv2 = df_tr[selected_transport_col].values.astype(float)

# -----------------------------
# 2) Wind series (Drake proxy)
# -----------------------------
if not wind_path.exists():
    raise FileNotFoundError(f'Missing wind file: {wind_path}')

df_wind = pd.read_csv(wind_path).copy()
if 'sector' in df_wind.columns:
    df_wind = df_wind[df_wind['sector'] == 'Atlantic'].copy()

if 'decimal_year' not in df_wind.columns:
    df_wind['decimal_year'] = df_wind['year'] + (df_wind['month'] - 0.5) / 12.0
df_wind = df_wind.sort_values('decimal_year').reset_index(drop=True)
t_wind2 = df_wind['decimal_year'].values.astype(float)
tau_x2 = df_wind['tau_x'].values.astype(float)

# -----------------------------
# 3) Drake EKE and KE series
# -----------------------------
def _drake_box_series_from_monthlies(monthly_dir, var_name, out_name, scale=1.0):
    rows = []
    files = sorted(monthly_dir.glob('*/*.nc'))
    if not files:
        return pd.DataFrame(columns=['year', 'month', 'decimal_year', out_name, 'n_valid'])

    for fp in files:
        try:
            yr = int(fp.parent.name)
            mo = int(fp.stem)
        except Exception:
            continue

        try:
            with xr.open_dataset(fp) as ds:
                if var_name not in ds:
                    continue

                lon_name = 'longitude' if 'longitude' in ds.coords else ('lon' if 'lon' in ds.coords else None)
                lat_name = 'latitude' if 'latitude' in ds.coords else ('lat' if 'lat' in ds.coords else None)
                if lon_name is None or lat_name is None:
                    continue

                lon = ds[lon_name].values.astype(float)
                lat = ds[lat_name].values.astype(float)
                fld = ds[var_name].values.astype(float)

                if np.nanmax(lon) > 180:
                    lon_min = DRAKE_LON_MIN % 360.0
                    lon_max = DRAKE_LON_MAX % 360.0
                else:
                    lon_min = DRAKE_LON_MIN
                    lon_max = DRAKE_LON_MAX

                m_lon = (lon >= lon_min) & (lon <= lon_max)
                m_lat = (lat >= DRAKE_LAT_MIN) & (lat <= DRAKE_LAT_MAX)

                if (not np.any(m_lon)) or (not np.any(m_lat)):
                    val_mean = np.nan
                    n_valid = 0
                else:
                    sub = fld[np.ix_(m_lat, m_lon)]
                    n_valid = int(np.isfinite(sub).sum())
                    val_mean = float(np.nanmean(sub) * scale) if n_valid > 0 else np.nan

                rows.append({
                    'year': yr,
                    'month': mo,
                    'decimal_year': yr + (mo - 0.5) / 12.0,
                    out_name: val_mean,
                    'n_valid': n_valid,
                })
        except Exception:
            continue

    out = pd.DataFrame(rows)
    if len(out):
        out = out.sort_values('decimal_year').reset_index(drop=True)
    return out

# Primary source: monthly altimetry maps in Drake box (m2 s-2 -> cm2 s-2)
df_eke_drake = _drake_box_series_from_monthlies(hp_dir, var_name='eke', out_name='eke_cm2', scale=1e4)
df_ke_drake = _drake_box_series_from_monthlies(hp_dir, var_name='ke', out_name='ke_cm2', scale=1e4)

# Fallback to Atlantic energy timeseries if needed
df_en = pd.read_csv(TRENDS_DIR / 'energy_field_timeseries.csv')
if 'sector' in df_en.columns and 'region' not in df_en.columns:
    df_en = df_en.rename(columns={'sector': 'region'})
df_en = df_en[df_en['region'] == 'Atlantic'].copy()
df_en = df_en.sort_values('decimal_year').reset_index(drop=True)

if df_eke_drake.empty:
    df_eke_drake = pd.DataFrame({
        'year': df_en['year'].values,
        'month': df_en['month'].values,
        'decimal_year': df_en['decimal_year'].values,
        'eke_cm2': (df_en['mean_eke'].values * 1e4),
        'n_valid': np.nan,
    })

if df_ke_drake.empty:
    df_ke_drake = pd.DataFrame({
        'year': df_en['year'].values,
        'month': df_en['month'].values,
        'decimal_year': df_en['decimal_year'].values,
        'ke_cm2': (df_en['mean_ke'].values * 1e4),
        'n_valid': np.nan,
    })

t_eke2 = df_eke_drake['decimal_year'].values.astype(float)
eke2 = df_eke_drake['eke_cm2'].values.astype(float)
t_ke2 = df_ke_drake['decimal_year'].values.astype(float)
ke2 = df_ke_drake['ke_cm2'].values.astype(float)

# -----------------------------
# 4) Trend helper and summaries
# -----------------------------
def _trend_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 8:
        return {'slope_yr': np.nan, 'intercept': np.nan, 'mk_p': np.nan}
    ts = stats.theilslopes(y[m], x[m], alpha=0.95)
    _, p = mann_kendall(y[m])
    return {'slope_yr': float(ts.slope), 'intercept': float(ts.intercept), 'mk_p': float(p)}

st_tr = _trend_stats(t_tr2, sv2)
st_wind = _trend_stats(t_wind2, tau_x2)
st_eke = _trend_stats(t_eke2, eke2)
st_ke = _trend_stats(t_ke2, ke2)

print('Section 9 computes diagnostics only (no additional figure).')
print('\nTrend summary (Theil-Sen slopes per year):')
print(f"  Transport altimetry ({selected_transport_col}): {st_tr['slope_yr']:+.4f} m² s⁻¹/yr, MK p={st_tr['mk_p']:.4f}")
print(f"  ADCP reference    : +{ADCP_SLOPE_SV_YR:.4f} Sv/yr, {ADCP_P_TEXT}")
print(f"  Wind (altimetry)  : {st_wind['slope_yr']:+.4f} tau_x/yr, MK p={st_wind['mk_p']:.4f}")
print(f"  EKE (Drake)       : {st_eke['slope_yr']:+.4f} cm2 s-2/yr, MK p={st_eke['mk_p']:.4f}")
print(f"  KE (Drake)        : {st_ke['slope_yr']:+.4f} cm2 s-2/yr, MK p={st_ke['mk_p']:.4f}")

Section 9 computes diagnostics only (no additional figure).

Trend summary (Theil-Sen slopes per year):
  Transport altimetry (transport_m2s): +155.4251 m² s⁻¹/yr, MK p=0.3259
  ADCP reference    : +0.0200 Sv/yr, p < 0.05
  Wind (altimetry)  : +0.0074 tau_x/yr, MK p=0.4882
  EKE (Drake)       : +1.7886 cm2 s-2/yr, MK p=0.0324
  KE (Drake)        : -0.0170 cm2 s-2/yr, MK p=0.9829


## 10. Altimetry KE and EKE Trends (No Additional Figure)

This section computes and prints **altimetry-based** KE and EKE trend statistics
for the Atlantic sector (Drake proxy), without creating a separate figure.

In [10]:
# Altimetry-based KE/EKE trends - statistics only
energy_alt_path = TRENDS_DIR / 'energy_field_timeseries.csv'
if not energy_alt_path.exists():
    raise FileNotFoundError(f'Missing altimetry energy file: {energy_alt_path}')

df_alt = pd.read_csv(energy_alt_path).copy()
if 'sector' in df_alt.columns:
    df_alt = df_alt[df_alt['sector'] == 'Atlantic'].copy()

if 'decimal_year' not in df_alt.columns:
    if 'year' in df_alt.columns and 'month' in df_alt.columns:
        df_alt['decimal_year'] = df_alt['year'] + (df_alt['month'] - 0.5) / 12.0
    else:
        raise ValueError('energy_field_timeseries.csv requires decimal_year or (year, month).')

df_alt = df_alt.sort_values('decimal_year').reset_index(drop=True)

# Convert m2 s-2 to cm2 s-2
alt_t = df_alt['decimal_year'].values.astype(float)
alt_eke = (df_alt['mean_eke'].values.astype(float) * 1e4)
alt_ke = (df_alt['mean_ke'].values.astype(float) * 1e4)

st_alt_eke = _trend_stats(alt_t, alt_eke)
st_alt_ke = _trend_stats(alt_t, alt_ke)

print('Section 10 computes altimetry trends only (no additional figure).')
print('\nAltimetry trend summary (Theil-Sen slopes per year):')
print(f"  EKE (Altimetry)     : {st_alt_eke['slope_yr']:+.4f} cm2 s-2/yr, MK p={st_alt_eke['mk_p']:.4f}")
print(f"  KE (Altimetry)      : {st_alt_ke['slope_yr']:+.4f} cm2 s-2/yr, MK p={st_alt_ke['mk_p']:.4f}")

Section 10 computes altimetry trends only (no additional figure).

Altimetry trend summary (Theil-Sen slopes per year):
  EKE (Altimetry)     : +1.3705 cm2 s-2/yr, MK p=0.0000
  KE (Altimetry)      : +0.9014 cm2 s-2/yr, MK p=0.0000
